# Showjumping SSL — Final Notebook

End-to-end **final** pipeline: cross-venue annotation -> automated fence detection -> venue-fixed SSL pretraining -> downstream heads + baselines -> sample-efficiency / ablation analysis -> all figures.

This picks up where `colab_milestone.ipynb` left off. It assumes the milestone already produced **segmented clips** on Drive (`data/clips/`). It then does everything the milestone deferred to "week 3/4":

- annotate fence boxes across **all** source videos (fixes the single-venue confound) + label outcomes
- fine-tune YOLOv8 for fences + a vertical/oxer CNN (recovers the proposal's automated detection)
- re-run SSL from **Kinetics init** with a **venue-balanced sampler** + **domain-adversarial** head
- train the type-conditioned outcome classifier + `d` regressor (frozen & fine-tuned)
- the three baselines, the sample-efficiency sweep, and the ablations
- every final figure (incl. the venue t-SNE that tells you whether the confound shrank)

**Two cells need you (human-in-the-loop):** section 3 fence annotation and section 4 outcome annotation. Everything else runs unattended.

**Runtime:** A100 recommended. SSL re-run ~15-20 min; the sweep (4 budgets x 3 seeds x 4 methods) is the longest unattended step.

**One-time edits:** set `GITHUB_REPO` and `DRIVE_DATA_ROOT` in section 0 to your own values.

## 0. Setup -- clone repo + mount Drive

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_REPO = 'https://github.com/bballhaus/showjumping-ssl.git'
DRIVE_DATA_ROOT = '/content/drive/MyDrive/CS131'
BRANCH = 'main'

from google.colab import drive
drive.mount('/content/drive')
Path(f'{DRIVE_DATA_ROOT}/data').mkdir(parents=True, exist_ok=True)
Path(f'{DRIVE_DATA_ROOT}/checkpoints').mkdir(parents=True, exist_ok=True)

REPO_DIR = Path('/content/project')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1',
                    GITHUB_REPO, str(REPO_DIR)], check=True)

for sub in ['data', 'checkpoints']:
    target = Path(f'{DRIVE_DATA_ROOT}/{sub}')
    link = REPO_DIR / sub
    if link.is_symlink() or link.exists():
        if link.is_dir() and not link.is_symlink():
            subprocess.run(['rm', '-rf', str(link)], check=True)
        else:
            link.unlink(missing_ok=True)
    link.symlink_to(target, target_is_directory=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('cwd:', os.getcwd())
print('data ->', os.readlink('data'))
print('checkpoints ->', os.readlink('checkpoints'))

## 1. Install dependencies (+ ultralytics for fence detection)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q ultralytics
!apt-get -qq install -y ffmpeg

import torch
print(f'torch: {torch.__version__} | cuda: {torch.cuda.is_available()}')
!nvidia-smi 2>&1 | head -10

## 2. Sanity check -- segmented clips are present

The final pipeline starts from the milestone's segmented clips. If this fails, run
`colab_milestone.ipynb` through clip segmentation first -- the clips persist on Drive.

In [ ]:
from pathlib import Path
from collections import Counter

clips = sorted(Path('data/clips').glob('*.mp4'))
assert clips, ('data/clips is empty. Run colab_milestone.ipynb through clip '
               'segmentation first (clips persist on Drive).')
by_v = Counter(p.stem.split('_', 1)[0] for p in clips)
print(f'{len(clips)} clips across {len(by_v)} source videos:')
for v, c in sorted(by_v.items()):
    print(f'  {v}: {c}')

## 3. Cross-venue fence annotation

The milestone's 39 boxes all came from **one** video -- that single-venue confound blocked
every approach-quality claim. Annotate boxes spanning **all** source videos here (target
~100 total). Scrub the slider to the frame where the fence is visible, drag a box, then click
Vertical / Oxer. Already-annotated clips are skipped, so you can stop and resume across sessions.

In [ ]:
!pip install -q ipympl ipywidgets
from google.colab import output
output.enable_custom_widget_manager()
%matplotlib widget

from src.preprocess.annotate_colab import ColabAnnotator

ann = ColabAnnotator('data/clips', 'data/annotations/fences.csv', limit=120)
ann.start()

## 4. Outcome annotation

Label outcomes {clean, knockdown, refusal} on the clips you fence-annotated (and as many
more as you can) so the downstream label table can join type / d / outcome. Scrub to watch
the jump, then click the outcome. Writes `data/annotations/outcomes.csv`.

In [ ]:
from src.preprocess.annotate_colab import OutcomeAnnotator

ann = OutcomeAnnotator('data/clips', 'data/annotations/outcomes.csv', limit=250)
ann.start()

## 5. Automated fence detection -- fine-tune YOLOv8 + vertical/oxer CNN

Recovers the proposal's automated path: export the hand-drawn boxes to YOLO format and
fine-tune YOLOv8 (single `fence` class), then train an ImageNet ResNet-18 to call
vertical vs. oxer from the cropped fence. Both persist to `checkpoints/`.

In [ ]:
from pathlib import Path
from src.preprocess.fence_yolo import export, train as yolo_train
from src.preprocess.fence_type_cnn import train as type_train

yaml_path = export(Path('data/clips'), Path('data/annotations/fences.csv'),
                   Path('data/fence_yolo'), val_frac=0.2)
best = yolo_train(yaml_path, weights='yolov8n.pt', epochs=100, imgsz=320, device=0)
print('fence YOLO best weights:', best)

type_train(Path('data/clips'), Path('data/annotations/fences.csv'),
           Path('checkpoints/fence_type.pt'), epochs=40, device='cuda')

## 6. YOLO horse + geometric d over all clips -> auto.csv

Runs COCO-YOLO horse detection + the takeoff-distance geometry on every clip. Hand-drawn
fence boxes take priority; clips without one fall back to the fine-tuned fence detector from
section 5. Also writes `by_video.csv` (clip -> source video) for venue grouping and the venue t-SNE.

In [ ]:
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from src.preprocess.run_pipeline import process_clip, load_fence_annotations
from src.preprocess.detect import HorseDetector
from src.preprocess.fence_yolo import FenceDetector

detector = HorseDetector(weights='yolov8n.pt', device='cuda')
fence_det = FenceDetector('checkpoints/fence_yolo/train/weights/best.pt', device='cuda')
fences = load_fence_annotations(Path('data/annotations/fences.csv'))
clips = sorted(Path('data/clips').glob('*.mp4'))
print(f'{len(clips)} clips, {len(fences)} hand boxes, auto-detect on for the rest')

rows = [process_clip(cp, detector, fences.get(cp.stem), fence_detector=fence_det)
        for cp in tqdm(clips, desc='YOLO + geometry', unit='clip')]
df = pd.DataFrame(rows)
df.to_csv('data/annotations/auto.csv', index=False)

pd.DataFrame({'clip_id': [p.stem for p in clips],
             'video': [p.stem.split('_', 1)[0] for p in clips]}
            ).to_csv('data/annotations/by_video.csv', index=False)

typed = (df['type'].astype(str).isin(('vertical', 'oxer'))).sum()
print(f'auto.csv: {len(df)} rows, {typed} typed, {df["d_meters"].notna().sum()} with d')

## 7. Merge labels -> labels.csv (one row per labeled clip: video, type, outcome, d)

In [ ]:
from pathlib import Path
from src.data.labels import build_labels

build_labels(Path('data/annotations/auto.csv'),
             Path('data/annotations/outcomes.csv'),
             Path('data/annotations/by_video.csv'),
             Path('data/annotations/labels.csv'))

## 8. SSL pretraining with the venue fix

Kinetics init (the milestone's fallback, since from-scratch collapsed) + venue-balanced
sampler + domain-adversarial head (`lambda_domain=0.3`) to actively suppress venue identity.
If the adversarial loss diverges, the arena-color jitter augmentation is already baked into
the dataset as the simpler fallback.

In [ ]:
from pathlib import Path
from src.ssl.train import train

train(
    clips_dir=Path('data/clips'),
    out_dir=Path('checkpoints'),
    epochs=15,
    batch_size=16,
    lr=1e-3,
    tau=0.1,
    lambda_order=0.3,
    n_workers=2,
    device='cuda',
    kinetics_init=True,    # proposal fallback (from-scratch collapsed in the milestone)
    balance_venues=True,   # venue-balanced sampler
    lambda_domain=0.3,     # domain-adversarial venue suppression
)

## 9. Embeddings (for t-SNE + retrieval)

In [ ]:
from pathlib import Path
from src.ssl.embed import embed_dir

embed_dir(Path('data/clips'), Path('checkpoints/encoder.pt'),
          Path('data/embeddings.npz'), device='cuda', batch_size=16)

## 10. Downstream heads -- frozen vs. fine-tuned, cross-venue split

`group_by_venue=True` holds out **entire source videos** for validation -- the honest
cross-venue test the milestone calls for. Returns accuracy / macro-F1 (per type) + d-MAE.

In [ ]:
from pathlib import Path
from src.downstream.train import run

labels = Path('data/annotations/labels.csv')
clips = Path('data/clips')

print('== frozen SSL encoder ==')
print(run(labels, clips, ckpt=Path('checkpoints/encoder.pt'), finetune=False,
          task='both', group_by_venue=True, epochs=30, device='cuda'))

print('== fine-tuned SSL encoder ==')
print(run(labels, clips, ckpt=Path('checkpoints/encoder.pt'), finetune=True,
          task='both', group_by_venue=True, epochs=30, device='cuda'))

## 11. Baselines -- type-only logreg, ImageNet-2D, from-scratch supervised

In [ ]:
from pathlib import Path
from src.downstream.train import run
from src.baselines.imagenet2d import ImageNet2DEncoder
from src.baselines.type_logreg import run as logreg_run

labels = Path('data/annotations/labels.csv')
clips = Path('data/clips')

print('type_logreg :', logreg_run(labels, group_by_venue=True))
print('imagenet2d  :', run(labels, clips, encoder=ImageNet2DEncoder(), finetune=False,
                           task='both', group_by_venue=True, epochs=30, device='cuda'))
print('from_scratch:', run(labels, clips, ckpt=None, kinetics_init=False, finetune=True,
                          task='both', group_by_venue=True, epochs=40, device='cuda'))

## 12. Sample-efficiency sweep -- the headline experiment

Accuracy / macro-F1 / d-MAE vs. label budget n in {25, 50, 100, 200}, for SSL vs. Kinetics
vs. from-scratch vs. ImageNet-2D, averaged over 3 seeds on a fixed cross-venue split. This is
the test of the proposal's hypothesis: SSL matches from-scratch with far fewer labels.

In [ ]:
from pathlib import Path
from src.eval.sample_efficiency import sweep

sweep(Path('data/annotations/labels.csv'), Path('data/clips'),
      Path('data/results/sample_efficiency.csv'),
      ckpt=Path('checkpoints/encoder.pt'),
      ns=(25, 50, 100, 200), seeds=(0, 1, 2),
      group_by_venue=True, epochs=30, device='cuda')

## 13. Ablations -- type conditioning + pretext tasks

Type conditioning on/off measures what explicit fence type buys. The pretext ablation
compares encoders trained with different objective mixes. To populate it, train variant
encoders first, e.g.:

```python
# from src.ssl.train import train
# train(Path('data/clips'), Path('checkpoints/info_only'), epochs=15, lambda_order=0.0,
#       kinetics_init=True, balance_venues=True, lambda_domain=0.3, device='cuda')   # InfoNCE only
# train(Path('data/clips'), Path('checkpoints/order_only'), epochs=15, lambda_order=1.0,
#       kinetics_init=True, balance_venues=True, lambda_domain=0.3, device='cuda')   # order-heavy
```

then add them to `ckpts=` below as `info_only=...`, `order_only=...`.

In [ ]:
from pathlib import Path
from src.eval.ablations import run_ablations

run_ablations(Path('data/annotations/labels.csv'), Path('data/clips'),
              Path('data/results/ablations.csv'),
              ckpts={'both': Path('checkpoints/encoder.pt')},
              type_ckpt=Path('checkpoints/encoder.pt'),
              seeds=(0, 1, 2), group_by_venue=True, epochs=30, device='cuda')

## 14. Build all figures

`make_figures` emits the training curve, d histogram, t-SNE (by type / outcome / d),
and the sample-efficiency curves. We then add the **venue t-SNE** -- the figure that tells
you whether the venue confound actually shrank -- and the detection grid.

In [ ]:
!python -m src.viz.make_figures \
    --log checkpoints/train_log.csv --csv data/annotations/auto.csv \
    --emb data/embeddings.npz --labels data/annotations/labels.csv \
    --results data/results/sample_efficiency.csv --out milestone/figures

from pathlib import Path
import pandas as pd
from src.viz.tsne import plot_tsne
from src.viz.detection_examples import render_clip, _save_grid
from src.preprocess.detect import Box, HorseDetector

OUT = Path('milestone/figures'); OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'det').mkdir(parents=True, exist_ok=True)

# Key figure: did the venue confound shrink after the balanced sampler + DANN?
plot_tsne(Path('data/embeddings.npz'), Path('data/annotations/by_video.csv'),
          color_by='video', out_path=OUT / 'tsne_video.png',
          title='t-SNE colored by source video (venue clustering)')

# Detection grid: horse box + fence box + computed d at the takeoff frame.
detector = HorseDetector(weights='yolov8n.pt', device='cuda')
fdf = pd.read_csv('data/annotations/fences.csv').head(4)
rendered = []
for _, row in fdf.iterrows():
    clip = Path('data/clips') / f"{row['clip_id']}.mp4"
    if not clip.exists():
        continue
    fb = Box(float(row['x1']), float(row['y1']), float(row['x2']), float(row['y2']), label='fence')
    pole = int(row['pole_count']) if pd.notna(row.get('pole_count')) else None
    out_jpg = OUT / 'det' / f"{row['clip_id']}.jpg"
    render_clip(clip, fb, pole, detector, out_jpg)
    if out_jpg.exists():
        rendered.append(out_jpg)
_save_grid(rendered, OUT / 'det_grid.png', cols=2)

print('figures:')
!ls -la milestone/figures/*.png

## 15. Preview figures inline

In [ ]:
from IPython.display import Image, display
from pathlib import Path

for p in sorted(Path('milestone/figures').glob('*.png')):
    print(p.name)
    display(Image(filename=str(p)))

## 16. Export figures to Drive (for the final report)

In [ ]:
import shutil, os
os.makedirs('/content/drive/MyDrive/CS131/final_export', exist_ok=True)
shutil.make_archive('/content/drive/MyDrive/CS131/final_export/figures', 'zip', 'milestone/figures')
print('zipped to /content/drive/MyDrive/CS131/final_export/figures.zip')